# Trabajo Práctico  - Calibración y Pose de la Cámara

## Resolución

## GRUPO: NOMBRE DEL GRUPO

# Integrantes
* integrante
* integrante

In [ ]:
import os
import importlib
import numpy as np
import alc

importlib.reload(alc)  # los cambios en alc.py no se ven sin reload
np.set_printoptions(precision=4, suppress=True)


### Ejercicio 1 - DLT ecuaciones formales

Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

Encontrar el vector $\mathbf{m}$ que resuelve la minimización de la ecuación 6.

***
Cada correspondencia entre un punto **real** del mundo $P_i = (U_i, V_i, W_i)$
y su proyección **en píxeles** de la imagen $(u_i, v_i)$ produce las 2 filas
$Q_{2i-1}, Q_{2i} \in \mathbb{R}^{2 \times 12}$ (filas 1 y 2 de la ecuación 6, ver más abajo).

Para $n$ puntos $Q$ es de $2n \times 12$ y se buscan las soluciones que satisfacen
$Q\mathbf{m} = \mathbf{0}$ (ecuación 6), es decir el sistema homogéneo.

Por el teorema del rango y la nulidad se tiene:

$$\operatorname{Rng}(Q) + \dim \operatorname{Nu}(Q) = \#\text{columnas} = 12$$

Si el sistema tuviera solución única, esa sería la trivial $\mathbf{m} = \mathbf{0}$
($\operatorname{Nu}(Q)$ $= \{\mathbf{0}\}$), que físicamente no sirve. Lo que buscamos es **unicidad de dirección**: que todas las soluciones
$\mathbf{m} \neq \mathbf{0}$ sean de la forma $c\,\mathbf{m}_1$, con $\mathbf{m}_1$
fijo — es decir, que no difieran más que en un factor de escala. El conjunto
solución debe ser una recta por el origen:

$$\dim(\operatorname{Nu}(Q)) = 1
\;\qquad\Longrightarrow\qquad \operatorname{Rng}(Q) = 12 - 1 = \boxed{11}$$

***

El rango de $Q$ está acotado por el menor entre su cantidad de filas y columnas:

$$\operatorname{Rng}(Q) \leq \min(2n,\,12)$$

Como probamos que $\operatorname{Rng}(Q) = 11$, se necesita

$$11 \leq \min(2n,\,12) \;\Longleftrightarrow\; 2n \geq 11 \;\Longleftrightarrow\; n \geq 5{,}5$$

Como $n$ debe ser entero, $2n \geq 12$, es decir:

$$n_{\min} = \boxed{6 \text{ puntos}}$$

Los puntos deben cubrir un espacio 3D en posición general: **no pueden ser coplanares entre
sí**. Si todos los puntos estuvieran en un plano, las filas de $Q$ serían redundantes entre sí
(las coordenadas dejarían de aportar información de la tercera dimensión), produciendo columnas
linealmente dependientes, reduciendo el rango de $Q$ y perdiendo la unicidad de la solución.
Es exactamente el caso degenerado del DLT 3D que motiva el método de Zhang para datos coplanares.

***
#### Encontrar el vector $\mathbf{m}$ tal que $\lVert Q\mathbf{m} \rVert$ sea mínimo (ec. 8)

Se busca el vector $\hat{\mathbf{m}}$ que resuelve

$$\min_{\lVert \mathbf{m} \rVert = 1} \; \lVert Q\mathbf{m} \rVert^2$$

Como la norma al cuadrado es un producto interno:

$$\lVert Q\mathbf{m} \rVert^2 = \mathbf{m}^\top\, (Q^\top Q)\, \mathbf{m}$$

Sea la descomposición en valores singulares $Q = U S V^\top$, con $S$ rectangular
($2n \times 12$) y valores singulares $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_{12} \geq 0$.

> **Nota:** $S^2$ **no está definida** porque $S$ es rectangular; en su lugar trabajamos con
> $\hat{S} = S^\top S$, que sí es cuadrada ($12 \times 12$) y diagonal. En $S$ los $\sigma_i$
> están en las primeras 12 posiciones de la diagonal; al hacer $S^\top S$ sobreviven solo los
> $\sigma_i^2$ en la diagonal.

$$\hat{S} \;=\; S^\top S \;=\; \operatorname{diag}(\sigma_1^2, \ldots, \sigma_{12}^2) \;\in\; \mathbb{R}^{12 \times 12}$$

Entonces $Q^\top Q$ se diagonaliza ortogonalmente (usando que $U$ es ortogonal, $U^\top U = I$):

$$Q^\top Q \;=\; (U S V^\top)^\top (U S V^\top)
\;=\; V S^\top \, U^\top U \, S V^\top
\;=\; V \, S^\top S \, V^\top
\;=\; V \, \hat{S} \, V^\top$$

Con $\mathbf{y} = V^\top \mathbf{m}$ y usando que $V$ ortogonal no cambia la norma
($\lVert \mathbf{y} \rVert = \lVert \mathbf{m} \rVert = 1$):

$$\mathbf{m}^\top V \hat{S} V^\top \mathbf{m}
\;=\; \mathbf{y}^\top \hat{S} \mathbf{y}
\;=\; \sum_{i=1}^{12} \sigma_i^2 \, y_i^2$$

$$\min_{\lVert \mathbf{m} \rVert = 1} \lVert Q\mathbf{m} \rVert^2
\;=\; \min_{\lVert \mathbf{y} \rVert = 1} \mathbf{y}^\top \hat{S} \mathbf{y}
\;=\; \min_{\lVert \mathbf{y} \rVert = 1} \sum_{i=1}^{12} \sigma_i^2 y_i^2$$

Como cada $\sigma_i^2 \geq \sigma_{12}^2$, pues están ordenados por la descomposición SVD, se tiene que:

$$\sum_{i=1}^{12} \sigma_i^2 y_i^2 \;\geq\; \sigma_{12}^2 \sum_{i=1}^{12} y_i^2 \;=\; \sigma_{12}^2$$

y la cota se alcanza eligiendo $y_i = 0$ para $i \neq 12$ y $y_{12} = 1$, es decir
$\mathbf{y} = \mathbf{e}_{12}$ (toda la masa de la restricción concentrada en el menor
valor singular).

#### Resultado

Volviendo a la variable original:

$$\min_{\lVert \mathbf{m} \rVert = 1} \lVert Q\mathbf{m} \rVert^2 = \sigma_{12}^2
\qquad\Longrightarrow\qquad \hat{\mathbf{m}} = V\,\mathbf{e}_{12} \;=\; \mathbf{v}_{12}$$

$$\boxed{\hat{\mathbf{m}} = \pm\, \mathbf{v}_{12} \quad \text{con} \quad
\lVert Q\hat{\mathbf{m}} \rVert_{\min} = \sigma_{12}}$$

es decir, el autovector **derecho** asociado al menor valor singular de $Q$
(autovalor $\sigma_{12}^2$ de $Q^\top Q$: la última **columna** de $V$). El signo es
arbitrario porque el sistema es homogéneo: $\pm\mathbf{v}_{12}$ definen la misma recta
solución.

**Interpretación.** Con $n = 6$ puntos exactos y en posición general, $\sigma_{12} = 0$
($\dim \operatorname{Nu}(Q) = 1$) y $\mathbf{v}_{12}$ es el vector nulo del sistema: la
solución exacta del DLT. Con datos reales (ruido), $\sigma_{12} > 0$ y $\mathbf{v}_{12}$ es
la mejor aproximación al núcleo en mínimos cuadrados homogéneos. Es lo que implementa
`resolverDLT3D` tomando la última fila de $V^\top$.

### Ejercicio 2 - Calibración con puntos 3D

a) Utilizando los datos de la carpeta $\texttt{calib/3d}$ calcular $\mathbf{R}$,  $\mathbf{t}$ y $\mathbf{K}$.

Calcular un nuevo set de parámetros para 

b) $\alpha_u = \alpha_v$ 

c) $K_{12}=0$



In [ ]:
# Ej 2-a: calibración con puntos 3D (Tsai/DLT) — datos de calib/3d
objPts = np.loadtxt(os.path.join("calib", "3d", "p_W_corners.txt"),
                    delimiter=",")   # (12, 3) en cm, NO coplanarios
detected = np.loadtxt(os.path.join("calib", "3d",
                                   "detected_corners.txt"))
# (210, 24): 1 fila por vista con las 12 esquinas intercaladas (u, v)
N_PTS, N_VISTAS = objPts.shape[0], detected.shape[0]

Ks, Rs, ts = [], [], []
err_repro = np.zeros(N_VISTAS)
for k in range(N_VISTAS):
    imgPts = detected[k].reshape(N_PTS, 2)   # (u, v) de esta vista
    Q = alc.obtenerQ3d(imgPts, objPts)       # (2n, 12): sistema ec. 6
    M = alc.resolverDLT3D(Q)                 # μ·K[R|t], con m34 >= 0
    Rj, tj, Kj = alc.devolverMatricesCalibracion3d(M)
    Ks.append(Kj); Rs.append(Rj); ts.append(tj)
    # QA por vista: error de reproyección con la M estimada
    err_max = 0.0
    for i in range(N_PTS):
        P = np.append(objPts[i], 1.0)
        w = alc.multiplicar(M, P)
        err_max = max(err_max, abs(w[0]/w[2] - imgPts[i, 0]),
                                abs(w[1]/w[2] - imgPts[i, 1]))
    err_repro[k] = err_max

# Intrínsecos: promedio sobre las vistas (la cámara es la misma)
K = np.mean(Ks, axis=0)
print("K promedio:\n", K)
print("std por vista de (αu, γ, u0):", np.std(Ks, axis=0)[0].round(2))
print("std por vista de (αv, v0):", np.std(Ks, axis=0)[1].round(2))
print("err reproyección: max %.2f px | promedio %.2f px"
      % (err_repro.max(), err_repro.mean()))

# Pose (R, t): escena estática → promedio entre vistas.
# El promedio lineal de rotaciones NO es rotación exacta: se
# re-ortonormaliza con el helper de alc.py (Gram-Schmidt corto).
Rm = np.mean(Rs, axis=0)
R = alc._ortonormalizarR(Rm[0], Rm[1], Rm[2])
t = np.mean(ts, axis=0)
print("R (promedio, re-ortonormal):\n", R)
print("t promedio (cm):", t)


K promedio:
 [[425.9314  -0.3365 357.0923]
 [  0.     423.5597 254.0073]
 [  0.       0.       1.    ]]
std por vista de (αu, γ, u0): [4.81 1.11 2.43]
std por vista de (αv, v0): [0.   4.66 3.51]
err reproyección: max 1.36 px | promedio 0.84 px
R (promedio, re-ortonormal):
 [[ 0.5471  0.1272 -0.8273]
 [-0.3369  0.9382 -0.0785]
 [ 0.7663  0.3217  0.5562]]
t promedio (cm): [-17.98   -13.8492  39.083 ]


**Resultados Ej 2-a**

Con las $210$ vistas del video y los $12$ puntos de la mira (que **no**
son coplanarios: su $Z$ toma los valores $0$, $-10.95$ y $-24.45$ cm; con
$n=12 \geq 6$ el rango $11$ de $Q$ se alcanza, cf. Ej 1-a), el pipeline
DLT 3D + descomposición recupera:

$$K_{\text{prom}} = \begin{pmatrix} 425.93 & -0.34 & 357.09 \\
0 & 423.56 & 254.01 \\ 0 & 0 & 1 \end{pmatrix}$$

- **Plausibilidad**: la resolución es $752\times 480$ (centro esperado
  $(376, 240)$), y el despeje da $u_0 = 357.1$, $v_0 = 254.0$: razonables.
  Con $f \approx 424$ px el FOV horizontal es $\approx 83^\circ$ (cámara
  ancha, coherente con un webcam/videollamada).
- **Estabilidad**: el desvío estándar por vista es $\sim 5$ px en los
  focales y $\sim 3$ px en el centro (escena estática; la dispersión
  proviene del ruido de detección de esquinas).
- **Error de reproyecci?n**: m?ximo $1.36$ px, promedio $0.84$ px
  sobre las $210$ vistas ? del mismo orden que el ruido de
  detecci?n ($\sim 0.5$ px por esquina): no hay error sistem?tico.
- **Skew**: $\gamma = K_{12} \approx -0.3$ px, prácticamente nulo; es el
  gancho natural para las imposiciones de b) y c).


In [ ]:
# Ej 2-b/c: nuevo set de parámetros imponiendo las restricciones del
# enunciado (d): αu = αv y K12 = 0, promediadas sobre las mismas vistas.
K_b = np.zeros((3, 3))
K_c = np.zeros((3, 3))
for k in range(N_VISTAS):
    imgPts = detected[k].reshape(N_PTS, 2)
    M = alc.resolverDLT3D(alc.obtenerQ3d(imgPts, objPts))
    Kb = alc.devolverMatricesCalibracion3d(M, au_av=True)[2]
    Kc = alc.devolverMatricesCalibracion3d(M, gamma_0=True)[2]
    K_b = K_b + Kb
    K_c = K_c + Kc
K_b = K_b / N_VISTAS
K_c = K_c / N_VISTAS
print("K_b (αu = αv):\n", K_b)
print("K_c (K12 = 0):\n", K_c)


K_b (αu = αv):
 [[423.5597  -0.3365 357.0923]
 [  0.     423.5597 254.0073]
 [  0.       0.       1.    ]]
K_c (K12 = 0):
 [[425.933    0.     357.0923]
 [  0.     423.5597 254.0073]
 [  0.       0.       1.    ]]


**Comparación de los tres sets de parámetros**

$$K_\text{base} = \begin{pmatrix} 425.9 & -0.3 & 357.1 \\
0 & 423.6 & 254.0 \\ 0 & 0 & 1 \end{pmatrix},
\quad
K_b = \begin{pmatrix} 423.6 & -0.3 & 357.1 \\
0 & 423.6 & 254.0 \\ 0 & 0 & 1 \end{pmatrix},
\quad
K_c = \begin{pmatrix} 425.9 & 0 & 357.1 \\
0 & 423.6 & 254.0 \\ 0 & 0 & 1 \end{pmatrix}$$

Como $\gamma$ de partida es casi nulo ($\approx -0.3$ px) y
$\alpha_u \approx \alpha_v$ ($\sim 0.5\%$ de diferencia), las
imposiciones cambian los parámetros por cantidades del orden del ruido
de detección — consistente con que ambas restricciones son
razonables para esta cámara.


### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


> ¿Qué rango debe tener $Q \in \mathbb{R}^{2n\times 9}$ para asegurar
> una solución única de $H$ a menos de un factor de escala?

**Demostración:**

Por la ecuación (11) del TP: $Q\mathbf{h} = \mathbf{0}$, con $Q$ de $2n \times 9$
y $\mathbf{h} = \vec{H}$ vectorizado de $9 \times 1$, donde $H$ es una homografía
válida de rango 3.

Por como está construido $\mathbf{h}$, es un vector no nulo
que pertenece al núcleo de $Q$, es decir $\mathbf{h} \in \mathcal{N}(Q)$ y
$\mathbf{h} \neq \mathbf{0}$.

Por lo tanto $\mathbf{h}$ es una solución no trivial del espacio nulo de $Q$.

Para que $\mathbf{h}$ sea una solución única a menos de un factor de escala, el conjunto de soluciones de $Q\mathbf{h}=0$ debe ser una recta que pasa por el origen: $\{c\,\mathbf{h}: c \in \mathbb{R}, c\neq 0\}$, es decir, $\dim\mathcal{N}(Q)=1$.

Por el teorema del rango y la nulidad, $\operatorname{rango}(Q) + \dim \mathcal{N}(Q) = 9$ 
($Q$ tiene 9 columnas). Dado que $\dim \mathcal{N}(Q) = 1$, entonces $$\operatorname{rango}(Q) = 9 - 1 = \boxed{8}$$

Como $\operatorname{rango}(Q) \le \min(2n,\,9)$, para alcanzar rango 8 se necesita
$2n \geq 8$, es decir, **al menos $n = 4$ puntos** por vista, y basta que exista un grupo de 4 en posición general (ninguna terna de esos 4 colineal). El tablero de 77 esquinas lo cumple de sobra.

>b) Obtener la matriz $\mathbf{K}$

>c) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.

### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---